In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/embeddinggemma-300M"
model1 = SentenceTransformer(model_id).to(device=device)

print(f"Device: {model1.device}")
print(model1)
print("Total number of parameters in the model:", sum([p.numel() for _, p in model1.named_parameters()]))

In [ ]:
# =========================
# Configuration
# =========================

INPUT_JSON_PATH = "path/to/input.json"

TEXT_FIELD_KEY = "<TEXT_FIELD>"
NAME_FIELD_KEY = "<ID_FIELD>"

OUTPUT_EMBEDDINGS_PATH = "embeddings.npy"
OUTPUT_NAMES_PATH = "names.npy"


In [ ]:
import json
import numpy as np


def clean_text(text: str) -> str:
    """Basic text cleanup applied before embedding."""
    return text.replace("</s>", "").strip()


with open(INPUT_JSON_PATH, "r") as f:
    data = json.load(f)



sentences = []
names = []

for key in sorted(data.keys()):
    item = data[key]

    if not isinstance(item, dict):
        continue

    if TEXT_FIELD_KEY not in item or NAME_FIELD_KEY not in item:
        continue

    text = clean_text(item[TEXT_FIELD_KEY])
    if text == "":
        continue

    sentences.append(text)
    names.append(item[NAME_FIELD_KEY])

print(f"Collected {len(sentences)} samples.")

In [ ]:
assert len(sentences) > 0, "No valid samples found."

embeddings = model.encode(
    sentences,
    normalize_embeddings=True,
    show_progress_bar=True,
)


np.save(OUTPUT_EMBEDDINGS_PATH, embeddings)
np.save(OUTPUT_NAMES_PATH, np.array(names))
